# RoadSense: fine-tune on local data (M7) and compare models (E1, E2)

Kaggle GPU notebook. Inputs to add:
* the RDD2022 dataset (as in notebook 01),
* `runs` from notebook 01 (save that notebook's output as a dataset, then add it here),
* your local set split by `scripts/split_local.py` (upload `ml/data/local` as a dataset).

Secret: `HF_TOKEN`. Outputs go to `/kaggle/working/results`; copy them into `ml/results/`.

In [ ]:
%pip install -q "ultralytics>=8.3" onnx onnxslim huggingface_hub
!git clone -q --depth 1 https://github.com/NinadPandith/Road-Sense.git /kaggle/working/Road-Sense || true
%cd /kaggle/working/Road-Sense/ml

In [ ]:
import glob
import os
from pathlib import Path

import yaml
from ultralytics import YOLO

WORK = Path("/kaggle/working")
RESULTS = WORK / "results"
RESULTS.mkdir(exist_ok=True)
HF_REPO = "YOUR-HF-USER/roadsense-models"  # <- change me


def find(pattern):
    hits = glob.glob(pattern, recursive=True)
    if not hits:
        raise FileNotFoundError(pattern)
    return Path(hits[0])


RDD_ALL = WORK / "data/rdd_all/data.yaml"  # re-create with notebook 01's prepare cell if missing
LOCAL = find("/kaggle/input/**/local/data.yaml")
BASE_SMALL = find("/kaggle/input/**/small_all/weights/best.pt")
BASE_NANO = find("/kaggle/input/**/nano_all/weights/best.pt")
print(LOCAL, BASE_SMALL, BASE_NANO)

## M7: fine-tune on RDD2022 + local frames

Training on both keeps what the model learned from RDD2022 while adapting to your city's roads.

In [ ]:
local = yaml.safe_load(LOCAL.read_text())
rdd = yaml.safe_load(RDD_ALL.read_text())
lroot, rroot = Path(local["path"]), Path(rdd["path"])
combined = {
    "path": "/",
    "train": [str(rroot / rdd["train"]), str(lroot / local["train"])],
    "val": [str(lroot / local["val"])],
    "test": [str(lroot / local["test"])],
    "names": local["names"],
}
COMBINED = WORK / "data/combined.yaml"
COMBINED.parent.mkdir(parents=True, exist_ok=True)
COMBINED.write_text(yaml.safe_dump(combined))


def finetune(base, name, epochs=30, batch=32):
    return YOLO(str(base)).train(
        data=str(COMBINED),
        epochs=epochs,
        imgsz=640,
        batch=batch,
        lr0=0.002,
        project=str(WORK / "runs"),
        name=name,
        exist_ok=True,
        seed=42,
        patience=10,
    )


finetune(BASE_SMALL, "local_ft_small")
finetune(BASE_NANO, "local_ft_nano", batch=64)

## E2: does local data help? (both models tested on the local test set)

In [ ]:
!python -m scripts.compare_models --data {LOCAL} --tag e2 --out {RESULTS} \
    --model rdd_only={BASE_SMALL} \
    --model rdd_plus_local={WORK}/runs/local_ft_small/weights/best.pt
print((RESULTS / "e2_models.md").read_text())

## E1: which model is best? (RDD2022 India test split)

Trains the two extra sizes needed for the comparison (medium, and the older YOLOv8s).

In [ ]:
RDD_INDIA = WORK / "data/rdd_india/data.yaml"
for weights, name in (("yolo11m.pt", "medium_all"), ("yolov8s.pt", "v8s_all")):
    if not (WORK / "runs" / name / "weights/best.pt").exists():
        YOLO(weights).train(
            data=str(RDD_ALL),
            epochs=100,
            imgsz=640,
            batch=16,
            project=str(WORK / "runs"),
            name=name,
            exist_ok=True,
            seed=42,
        )

In [ ]:
!python -m scripts.compare_models --data {RDD_INDIA} --tag e1 --out {RESULTS} \
    --model yolo11n={BASE_NANO} \
    --model yolo11s={BASE_SMALL} \
    --model yolo11m={WORK}/runs/medium_all/weights/best.pt \
    --model yolov8s={WORK}/runs/v8s_all/weights/best.pt
print((RESULTS / "e1_models.md").read_text())

## Publish the fine-tuned models (version v2-local)

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!python -m scripts.export_and_upload --small {WORK}/runs/local_ft_small/weights/best.pt --nano {WORK}/runs/local_ft_nano/weights/best.pt --repo {HF_REPO} --version v2-local --out {WORK}/export

## Next
1. Download `/kaggle/working/results` into `ml/results/`.
2. Set `MODEL_VERSION=v2-local` on the backend; copy the new `yolo-nano.onnx` for Drive mode.
3. From `ml/`: `python -m scripts.build_results_json` and commit `frontend/public/results/`.